# Lab: Regression Analysis

### Before you start:

* Read the README.md file
* Comment as much as you can and use the resources (README.md file) 

Happy learning!

## Challenge 1
I work at a coding bootcamp, and I have developed a theory that the younger my students are, the more often they are late to class. In order to test my hypothesis, I have collected some data in the following table:

| StudentID | Age | Tardies |
|--------|-----|------------|
| 1      | 17  | 10         |
| 2      | 51  | 1          |
| 3      | 27  | 5          |
| 4      | 21  | 9         |
| 5      | 36  |  4         |
| 6      | 48  |  2         |
| 7      | 19  |  9         |
| 8      | 26  | 6          |
| 9      | 54  |  0         |
| 10     | 30  |  3         |

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns

Use this command to create a dataframe with the data provided in the table. 
~~~~
student_data = pd.DataFrame({'Age': [17,51,27,21,36,48,19,26,54,30], 'Tardies': [10,1,5,9,4,2,9,6,0,3]})
~~~~

In [ ]:
student_data = pd.DataFrame({'Age': [17,51,27,21,36,48,19,26,54,30], 'Tardies': [10,1,5,9,4,2,9,6,0,3]})
student_data

Draw a dispersion diagram (scatter plot) for the data.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(student_data['Age'], student_data['Tardies'])
plt.xlabel('Age')
plt.ylabel('Tardies')
plt.title('Age vs. Number of Tardies')
plt.show()

Do you see a trend? Can you make any hypotheses about the relationship between age and number of tardies?

*Yes, there's a clear downward trend: older students tend to have fewer
tardies, and younger students tend to have more. This supports the
hypothesis that younger students are late more often -- the relationship
looks strongly negative and fairly linear.*

Calculate the covariance and correlation of the variables in your plot. What is the difference between these two measures? Compare their values. What do they tell you in this case? Add your responses as comments after your code.

In [ ]:
covariance = np.cov(student_data['Age'], student_data['Tardies'])[0, 1]
correlation = student_data['Age'].corr(student_data['Tardies'])

print(f"Covariance: {covariance:.3f}")
print(f"Correlation: {correlation:.3f}")

# Covariance tells us the DIRECTION of the relationship (negative here,
# confirming that as Age goes up, Tardies goes down), but its magnitude
# is not easily interpretable on its own -- it depends on the units and
# scale of both variables (here, years and count of tardies).
#
# Correlation is the standardized version of covariance (always between
# -1 and 1), so it also tells us the STRENGTH of the relationship, not
# just the direction. A correlation of -0.939 indicates a very strong
# negative linear relationship -- much easier to interpret at a glance
# than the raw covariance of -45.57.

Build a regression model for this data. What will be your outcome variable? What type of regression are you using? Add your responses as comments after your code.

In [ ]:
# Outcome variable: Tardies (what we're trying to predict/explain).
# Predictor variable: Age.
# Since we have one continuous outcome and one continuous predictor,
# and we're modeling a straight-line relationship between them, this
# calls for a SIMPLE LINEAR REGRESSION.

model_1 = smf.ols('Tardies ~ Age', data=student_data).fit()
print(model_1.summary())

Plot your regression model on your scatter plot.

In [1]:
plt.figure(figsize=(8, 6))
plt.scatter(student_data['Age'], student_data['Tardies'], label='Data')
plt.plot(student_data['Age'], model_1.predict(student_data['Age']), color='red', label='Regression line')
plt.xlabel('Age')
plt.ylabel('Tardies')
plt.title('Age vs. Tardies with Fitted Regression Line')
plt.legend()
plt.show()

Interpret the results of your model. What can conclusions can you draw from your model and how confident in these conclusions are you? Can we say that age is a good predictor of tardiness? Add your responses as comments after your code.

*The model has an R-squared of 0.882, meaning Age explains about 88% of
the variation in Tardies -- a very strong fit for only 10 data points.
The slope (-0.243) is highly significant (p < 0.001), confirming that
each additional year of age is associated with about 0.24 fewer tardies
on average. Given both the strength of the fit and the statistical
significance, yes -- age does appear to be a good predictor of
tardiness in this sample. That said, with only 10 students this is a
small sample, so I would be cautious about generalizing too strongly
without more data (and this is observational data, so it shows
association, not necessarily causation).*

## Challenge 2
For the second part of this lab, we will use the vehicles.csv data set. You can find a copy of the dataset in the git hub folder. This dataset includes variables related to vehicle characteristics, including the model, make, and energy efficiency standards, as well as each car's CO2 emissions. As discussed in class the goal of this exercise is to predict vehicles' CO2 emissions based on several independent variables. 

In [ ]:
# Import any libraries you may need & the data
vehicles = pd.read_csv("vehicles.csv")

Let's use the following variables for our analysis: Year, Cylinders, Fuel Barrels/Year, Combined MPG, and Fuel Cost/Year. We will use 'CO2 Emission Grams/Mile' as our outcome variable. 

Calculate the correlations between each of these variables and the outcome. Which variable do you think will be the most important in determining CO2 emissions? Which provides the least amount of helpful information for determining CO2 emissions? Add your responses as comments after your code.

In [ ]:
columns_of_interest = ['Year', 'Cylinders', 'Fuel Barrels/Year', 'Combined MPG',
                        'Fuel Cost/Year', 'CO2 Emission Grams/Mile']

correlations = vehicles[columns_of_interest].corr()['CO2 Emission Grams/Mile'].drop('CO2 Emission Grams/Mile')
correlations = correlations.sort_values(key=abs, ascending=False)
print(correlations)

# Fuel Barrels/Year has by far the strongest correlation with CO2 emissions
# (~0.99) -- it's essentially a direct measure of how much fuel the car
# burns, which is the direct source of CO2 emissions, so this makes
# physical sense.
#
# Year has the weakest correlation (~-0.22) -- the model year gives very
# little direct information about emissions once you already know how
# much fuel a car burns and how efficient it is.

Build a regression model for this data. What type of regression are you using? Add your responses as comments after your code.

In [ ]:
# Since we have multiple predictor variables and one continuous outcome
# (CO2 Emission Grams/Mile), this calls for a MULTIPLE LINEAR REGRESSION.

# Renaming columns to avoid spaces/slashes, which aren't valid in
# statsmodels' formula syntax.
vehicles_model_data = vehicles.rename(columns={
    'Fuel Barrels/Year': 'Fuel_Barrels_Year',
    'Combined MPG': 'Combined_MPG',
    'Fuel Cost/Year': 'Fuel_Cost_Year',
    'CO2 Emission Grams/Mile': 'CO2_Emission_Grams_Mile'
})

model_2 = smf.ols(
    'CO2_Emission_Grams_Mile ~ Year + Cylinders + Fuel_Barrels_Year + Combined_MPG + Fuel_Cost_Year',
    data=vehicles_model_data
).fit()

Print your regression summary, and interpret the results. What are the most important varibles in your model and why? What can conclusions can you draw from your model and how confident in these conclusions are you? Add your responses as comments after your code.

In [ ]:
print(model_2.summary())

# R-squared = 0.981 -- this model explains 98.1% of the variance in CO2
# emissions, an excellent fit.
#
# All 5 predictors are statistically significant (p < 0.001). Looking at
# the coefficients: Fuel_Barrels_Year has by far the largest coefficient
# (~19.05) and is the strongest driver of predicted emissions, consistent
# with it having the highest correlation with the outcome above. Combined_MPG
# has a negative coefficient (~-3.04), which makes sense -- more fuel-efficient
# cars (higher MPG) emit less CO2 per mile.
#
# Given the very high R-squared and highly significant p-values across the
# board (and this being real-world, non-experimental data with 35,952
# observations), I'm fairly confident in these results as a description of
# the associations in this dataset. The large condition number in the
# summary output is worth noting though -- it suggests some multicollinearity
# among the predictors (e.g. Fuel_Barrels_Year and Fuel_Cost_Year are likely
# correlated with each other), so individual coefficients should be
# interpreted with some caution even though the overall model fits very well.

## Bonus Challenge: Error Analysis

I am suspicious about the last few parties I have thrown: it seems that the more people I invite the more people are unable to attend. To know if my hunch is supported by data, I have decided to do an analysis. I have collected my data in the table below, where X is the number of people I invited, and Y is the number of people who attended. 

|  X |  Y |
|----|----|
| 1  |  1 |
| 3  |  2 |
| 4  |  4 |
| 6  |  4 |
| 8  |  5 |
| 9  |  7 |
| 11 |  8 |
| 14 |  13 |

We want to know if the relationship modeled by the two random variables is linear or not, and therefore if it is appropriate to model it with a linear regression. 
First, build a dataframe with the data. 

In [ ]:
party_data = pd.DataFrame({'X': [1,3,4,6,8,9,11,14], 'Y': [1,2,4,4,5,7,8,13]})
party_data

Draw a dispersion diagram (scatter plot) for the data, and fit a regression line.

In [ ]:
model_party = smf.ols('Y ~ X', data=party_data).fit()

plt.figure(figsize=(8, 6))
plt.scatter(party_data['X'], party_data['Y'], label='Data')
plt.plot(party_data['X'], model_party.predict(party_data['X']), color='red', label='Regression line')
plt.xlabel('People Invited (X)')
plt.ylabel('People Who Attended (Y)')
plt.title('Invitations vs. Attendance')
plt.legend()
plt.show()

print(model_party.summary())

What do you see? What does this plot tell you about the likely relationship between the variables? Print the results from your regression.

*The points follow a fairly clear increasing straight-line pattern, and
the regression fits reasonably well (R-squared = 0.932). This actually
runs counter to the initial hunch -- the data looks roughly linear, with
attendance increasing steadily as more people are invited, rather than
showing a worsening no-show rate at higher invite counts.*

Do you see any problematic points, or outliers, in your data? Remove these points and recalculate your regression. Print the new dispersion diagram with your new model and the results of your model. 

In [2]:
# Using Cook's distance to identify influential points/outliers.
influence = model_party.get_influence()
cooks_d = influence.cooks_distance[0]

for x, y, cd in zip(party_data['X'], party_data['Y'], cooks_d):
    print(f"X={x}, Y={y}, Cook's distance={cd:.4f}")

# The point (X=14, Y=13) has a Cook's distance of ~2.05 -- dramatically
# higher than every other point (all below 0.17). This is our largest
# party by far, and it's disproportionately influencing the regression
# line. We'll remove it and refit.

party_data_clean = party_data[party_data['X'] != 14]

model_party_clean = smf.ols('Y ~ X', data=party_data_clean).fit()

plt.figure(figsize=(8, 6))
plt.scatter(party_data_clean['X'], party_data_clean['Y'], label='Data (outlier removed)')
plt.plot(party_data_clean['X'], model_party_clean.predict(party_data_clean['X']), color='red', label='Regression line')
plt.xlabel('People Invited (X)')
plt.ylabel('People Who Attended (Y)')
plt.title('Invitations vs. Attendance (outlier removed)')
plt.legend()
plt.show()

print(model_party_clean.summary())

What changed? Based on the results of the two models and your graphs, what can you say about the form of the data with the problematic point and without it?

*After removing the (14, 13) outlier, R-squared actually improved
slightly (0.932 -> 0.943), but more importantly the model itself changed
in an interesting way: the slope dropped from ~0.85 to ~0.68, and the
intercept moved from a physically odd -0.44 (predicting a negative
number of attendees at X=0) to a much more sensible ~0.32 (close to 0
attendees when 0 people are invited).

Without the large outlier party, the remaining data still supports a
linear relationship reasonably well, but with a gentler slope -- meaning
that, ignoring that one big outlier event, attendance grows a bit more
slowly relative to invitations than the full dataset suggested. This is
a good reminder that a single large, high-leverage data point can visually
"anchor" a regression line and make the fit look stronger and steeper
than the more typical, smaller parties actually suggest.*